In [3]:
# Sel ini menghasilkan TIGA dataset cabang (kota) terpisah untuk Tugas Mandiri Pertemuan 3
import numpy as np
import pandas as pd

kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-08-01", "2026-08-31", freq="D")

cabang_kota = {"Magelang": 101, "Yogyakarta": 202, "Semarang": 303}

for kota, seed in cabang_kota.items():
    np.random.seed(seed)  # seed berbeda tiap kota agar datanya bervariasi, namun tetap konsisten/reproducible
    n = 200
    data_cabang = {
        "order_id": [f"{kota[:3].upper()}-{2000 + i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "unit_terjual": np.random.randint(1, 8, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    }
    df_cabang = pd.DataFrame(data_cabang)
    df_cabang["kota"] = kota
    nama_file = f"transaksi_{kota.lower()}.csv"
    df_cabang.to_csv(nama_file, index=False)
    print(f"Berkas '{nama_file}' berhasil dibuat: {df_cabang.shape[0]} baris")

print("\nKetiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.")

Berkas 'transaksi_magelang.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_yogyakarta.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_semarang.csv' berhasil dibuat: 200 baris

Ketiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.


In [6]:
# Membuat direktori raw dan processed
!hdfs dfs -mkdir -p /user/vandrasembiring/ecommerce/raw
!hdfs dfs -mkdir -p /user/vandrasembiring/ecommerce/processed

# Menampilkan hasilnya (menggunakan -R untuk melihat isi subdirektori)
!hdfs dfs -ls -R /user/vandrasembiring/ecommerce

drwxr-xr-x   - vandrasembiring supergroup          0 2026-09-09 20:24 /user/vandrasembiring/ecommerce/processed
drwxr-xr-x   - vandrasembiring supergroup          0 2026-09-09 20:32 /user/vandrasembiring/ecommerce/raw


In [7]:
# Mengunggah ketiga berkas CSV ke HDFS
!hdfs dfs -put transaksi_magelang.csv transaksi_yogyakarta.csv transaksi_semarang.csv /user/vandrasembiring/ecommerce/raw/

# Menampilkan isi direktori raw untuk membuktikan berkas terunggah beserta ukurannya
!hdfs dfs -ls /user/vandrasembiring/ecommerce/raw

Found 3 items
-rw-r--r--   1 vandrasembiring supergroup      12329 2026-09-09 20:32 /user/vandrasembiring/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 vandrasembiring supergroup      12154 2026-09-09 20:32 /user/vandrasembiring/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 vandrasembiring supergroup      12684 2026-09-09 20:32 /user/vandrasembiring/ecommerce/raw/transaksi_yogyakarta.csv


In [8]:
import io
import subprocess
import pandas as pd

files = [
    "/user/vandrasembiring/ecommerce/raw/transaksi_magelang.csv",
    "/user/vandrasembiring/ecommerce/raw/transaksi_yogyakarta.csv",
    "/user/vandrasembiring/ecommerce/raw/transaksi_semarang.csv"
]

def baca_csv(hdfs_path):
    cmd = f"hdfs dfs -cat {hdfs_path}"
    csv_data = subprocess.check_output(cmd, shell=True)
    return pd.read_csv(io.BytesIO(csv_data))

df_list = [baca_csv(f) for f in files]
df_combined = pd.concat(df_list, ignore_index=True)

print(df_combined['kota'].value_counts())

kota
Magelang      200
Yogyakarta    200
Semarang      200
Name: count, dtype: int64


In [15]:
# 1. Menambahkan kolom total_pendapatan
df_combined['total_pendapatan'] = df_combined['unit_terjual'] * df_combined['harga_satuan']

# 2. Membuat tabel ringkasan per kota dan kategori
ringkasan = df_combined.groupby(['kota', 'kategori'])['total_pendapatan'].sum().reset_index()

# Menampilkan sedikit cuplikan hasil ringkasan
display(ringkasan.head())

# 3. Menyimpan hasil ke disk lokal
df_combined.to_csv("data_gabungan_bersih.csv", index=False)
ringkasan.to_csv("ringkasan_kota_kategori.csv", index=False)

# Mengunggah kedua hasil tersebut ke HDFS (direktori processed)
!hdfs dfs -put data_gabungan_bersih.csv ringkasan_kota_kategori.csv /user/vandrasembiring/ecommerce/processed/

# Verifikasi file sudah ada di folder processed
!hdfs dfs -ls /user/vandrasembiring/ecommerce/processed

,kota,kategori,total_pendapatan
0,Magelang,Elektronik,18775000
1,Magelang,Fashion,27750000
2,Magelang,Kesehatan & Kecantikan,17375000
3,Magelang,Makanan & Minuman,17525000
4,Magelang,Rumah Tangga,12200000


Found 2 items
-rw-r--r--   1 vandrasembiring supergroup      41257 2026-09-09 20:51 /user/vandrasembiring/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 vandrasembiring supergroup        530 2026-09-09 20:51 /user/vandrasembiring/ecommerce/processed/ringkasan_kota_kategori.csv
